email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

## LangGraph `Command`：同时控制状态更新与流程跳转

`Command` 是 LangGraph 中的核心类型，允许节点或工具在一次返回中**同时**完成两件事：更新 graph state 和控制下一步流程。

```python
from langgraph.types import Command

Command(
    update=...,   # 更新 graph state（dict 或消息列表）
    goto=...,     # 跳转到指定节点（单个、多个或带参数的 Send）
    resume=...,   # 恢复被 interrupt() 暂停的执行（用于 human-in-the-loop）
    graph=...,    # 跨图跳转时指定目标 graph，如 Command.PARENT
)
```

| 参数 | 典型用法 |
|------|---------|
| `update` | 工具修改 state，如更新 `authenticated` 标志并注入 `ToolMessage` |
| `goto` | 工具决定下一个执行节点，支持并行跳转 |
| `resume` | 向被中断节点传递用户决策（approve / reject） |
| `graph` | 子图返回父图节点：`graph=Command.PARENT` |

下方的 `authenticate` 工具使用 `Command(update={...})` 直接将认证结果写入 state，并同时注入 `ToolMessage` 作为消息历史。

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

I found one new email from Jane (jane@example.com) asking to grab a coffee next week.

Here’s a ready-to-send reply you can use. It’s friendly and keeps the ball in Jane’s court for timing.

Subject: Re: Coffee next week
Body:
Hi Jane,

That sounds great! I’d love to catch up. I’m around next week—are you free on Tuesday or Thursday afternoon, or Friday morning? If you have a preferred time or place, let me know and we can meet there.

Looking forward to it!
Julie

Would you like me to send this now, or would you prefer a slightly different version or different times? I can also propose a couple of specific slots (e.g., Tue 2–4 pm, Thu 11 am–1 pm, Fri 9–11 am) if you’d like.


In [9]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [10]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi Jane,

That sounds great! I’d love to catch up. I’m around next week—are you free on Tuesday or Thursday afternoon, or Friday morning? If you have a preferred time or place, let me know and we can meet there.

Looking forward to it!
Julie


In [11]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Email sent successfully to jane@example.com with the subject "Re: Coffee next week."

Body was:
Hi Jane,

That sounds great! I’d love to catch up. I’m around next week—are you free on Tuesday or Thursday afternoon, or Friday morning? If you have a preferred time or place, let me know and we can meet there.

Looking forward to it!
Julie

Would you like me to set a reminder to follow up if there’s no reply in a couple of days, or draft a couple of precise time-slot options for you to send later?


In [12]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='2c8a5b26-6fe6-4520-bf14-fc8c21c31683'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 151, 'total_tokens': 309, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DSOinYV56LQWoVpDVgf0hZ6X9ZHdH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d6d9e-4809-7521-90ef-7bf2915f946c-0', tool_calls=[{'name': 'authenticate', 'args': {'email': 'julie@example.com', 'password': 'password123'}, 'id': 'call_GzuV7XisC7cY7yrBZeKCjWmj', 'type': 'tool_call